In [ ]:
import openai
from qdrant_client import QdrantClient

from openai import OpenAI

openai_client = OpenAI(api_key="")


Embedding Function

In [14]:
def get_embedding(text, model="text-embedding-3-small"):
    # Use the openai_client instance created in Cell 1
    response = openai_client.embeddings.create(
        input=text,
        model=model,
    )
    return response.data[0].embedding

Retreival Function

In [22]:
qdrant_client = QdrantClient(url="http://localhost:6333")

def retrieve_data(query,qdrant_client, k=5):
    query_embedding = get_embedding(query)
    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-00",
        query=query_embedding,
        limit=k,
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["description"])
        retrieved_context_ratings.append(result.payload["rating"])
        similarity_scores.append(result.score)

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "retrieved_context_ratings": retrieved_context_ratings,
        "similarity_scores": similarity_scores,
    }
        

In [23]:
retrieved_context = retrieve_data("What are some goood kid movies?", qdrant_client, k=10)

In [ ]:
retrieved_context

In [26]:
def process_context(context:dict):

    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context

In [27]:
preprocessed_context = process_context(retrieved_context)

In [29]:
print(preprocessed_context)

- ID: B00ENYKO82, rating: 5, description: Loved this kid-friendly movie W e   l o v e d   t h i s   m o v i e   a s   a   f a m i l y .     F u n n y ,   w i t h o u t   a l l   o f   t h e   c r u d e   h u m o r   t h a t   t e n d s   t o   g o   o n   w i t h   e v e n   & # 3 4 ; k i d - f r i e n d l y & # 3 4 ;   m o v i e s .     O n e   o f   o u r   f a v e s   f o r   s u r e !
- ID: B07FN83ZK1, rating: 5, description: I'm a sucker for these sappy movies I   l i k e   a n y   m o v i e   H i l a r i e   B u r t o n   i s   i n ,   I   r e a l l y   l i k e   h e r .   T h i s   m o v i e   i s   l i k e   m o s t     H a l l m a r k   m o v i e s   a n d   t h e y   a r e   m y   g u i l t y   p l e a s u r e .   I   l i k e d   s e e i n g   a l l   t h e   f o o d   i n   t h e   m o v i e   a n d   l o v e   h o w   t h e   c h a r a c t e r s   c o m e   t o g e t h e r .   I ' v e   w a t c h e d   t h i s   m o v i e   s e v e r a l   t i m e s   a n d   I   a m   s u 

Prepare prompt

In [30]:
def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a movie recommendation assistant.

You will be given a question and a list of context.

Instructtions:
- You need to answer the question based on the provided context only.
- Never use word context and refer to it as the available products.

Context:
{preprocessed_context}

Question:
{question}
"""

    return prompt

In [31]:
prompt = build_prompt(preprocessed_context, "What are some good kid movies?")

In [32]:
print(prompt)


You are a movie recommendation assistant.

You will be given a question and a list of context.

Instructtions:
- You need to answer the question based on the provided context only.
- Never use word context and refer to it as the available products.

Context:
- ID: B00ENYKO82, rating: 5, description: Loved this kid-friendly movie W e   l o v e d   t h i s   m o v i e   a s   a   f a m i l y .     F u n n y ,   w i t h o u t   a l l   o f   t h e   c r u d e   h u m o r   t h a t   t e n d s   t o   g o   o n   w i t h   e v e n   & # 3 4 ; k i d - f r i e n d l y & # 3 4 ;   m o v i e s .     O n e   o f   o u r   f a v e s   f o r   s u r e !
- ID: B07FN83ZK1, rating: 5, description: I'm a sucker for these sappy movies I   l i k e   a n y   m o v i e   H i l a r i e   B u r t o n   i s   i n ,   I   r e a l l y   l i k e   h e r .   T h i s   m o v i e   i s   l i k e   m o s t     H a l l m a r k   m o v i e s   a n d   t h e y   a r e   m y   g u i l t y   p l e a s u r e .   I   l 

In [34]:

def generate_answer(prompt):

    response = openai.chat.completions.create(
        model="gpt-5-nano",
        messages=[{"role": "system", "content": prompt}],
        reasoning_effort="minimal"
    )

    return response.choices[0].message.content

In [33]:
def rag_pipeline(question, top_k=5):

    qdrant_client = QdrantClient(url="http://localhost:6333")

    retrieved_context = retrieve_data(question, qdrant_client, top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    return answer